# Proyecto IA Traductor — Notebook maestro

**Traducción de audio ES→EN con clonación de voz zero-shot**

Este notebook es el entregable maestro. Ejecuta el pipeline completo:

```text
Audio ES → Whisper-small → MarianMT base → XTTS-v2 → ECAPA-TDNN
```

Alcance de esta ejecución:

- Dataset B: 20 muestras del split oficial `test` de FLEURS `es_419`.
- Traducción: MarianMT **base**. El checkpoint de 10 000 pares se documenta como experimento, pero no se usa por defecto porque obtuvo BLEU/chrF inferiores.
- No se reentrena NMT: el experimento de 10k se carga desde evidencia sellada.
- Se verifica Gradio y se prueban entradas inválidas.
- MOS no se realiza; se declara como limitación.


## 1. Repositorio, trazabilidad y RUN_ID

En Colab se clona el repositorio. En local se usa el repositorio actual.
Cada ejecución crea un directorio nuevo para no sobrescribir resultados anteriores.


In [1]:
import json
import os
import platform
import random
import subprocess
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings("ignore")
NOTEBOOK_STARTED = time.perf_counter()
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Setrox1250/Proyecto-IA-Traductor.git"

if IN_COLAB:
    PROJECT_ROOT = Path("/content/Proyecto-IA-Traductor")
    if not (PROJECT_ROOT / ".git").exists():
        PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

def git(*args):
    return subprocess.check_output(["git", *args], cwd=PROJECT_ROOT, text=True).strip()

GIT_BRANCH = git("branch", "--show-current")
GIT_COMMIT = git("rev-parse", "HEAD")
GIT_STATUS = git("status", "--short")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
RUN_ROOT = PROJECT_ROOT / "outputs" / "final_eval" / f"master_{RUN_ID}"
RUN_ROOT.mkdir(parents=True, exist_ok=False)

print("PROJECT_ROOT:", PROJECT_ROOT.name)
print("RUN_ID:", RUN_ID)
print("GIT_BRANCH:", GIT_BRANCH)
print("GIT_COMMIT:", GIT_COMMIT)
print("GIT_STATUS:\n", GIT_STATUS or "(limpio)")


PROJECT_ROOT: IA-Proyecto
RUN_ID: 20261104_082658
GIT_BRANCH: main
GIT_COMMIT: e2df5999daf5a0635c8986794742fef35f179b1d
GIT_STATUS:
 (limpio)


## 2. Entorno y dependencias

En local se usa el entorno `.venv-xtts` ya validado. En Colab se instalan los
requisitos pineados. XTTS-v2 exige aceptar la licencia CPML mediante
`COQUI_TOS_AGREED=1`.


In [2]:
os.environ["COQUI_TOS_AGREED"] = "1"

if IN_COLAB:
    requirements = [
        "requirements/base.txt",
        "requirements/dataset.txt",
        "requirements/asr.txt",
        "requirements/translation.txt",
        "requirements/voice_cloning.txt",
    ]
    for requirement in requirements:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-r", requirement],
            cwd=PROJECT_ROOT,
            check=True,
        )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "matplotlib==3.11.2"],
        check=True,
    )

import importlib.metadata as metadata

import datasets
import gradio
import numpy as np
import pandas as pd
import soundfile
import speechbrain
import torch
import transformers
from IPython.display import display

def installed_version(*names):
    for name in names:
        try:
            return metadata.version(name)
        except metadata.PackageNotFoundError:
            continue
    return "no instalado"

CUDA_AVAILABLE = torch.cuda.is_available()
if not CUDA_AVAILABLE:
    raise RuntimeError("Este notebook requiere GPU. Active un runtime con CUDA.")

environment = {
    "run_id": RUN_ID,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "operating_system": platform.platform(),
    "gpu": torch.cuda.get_device_name(0),
    "vram_total_gib": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 3),
    "torch": torch.__version__,
    "torch_cuda_available": CUDA_AVAILABLE,
    "cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "coqui_tts": installed_version("coqui-tts", "TTS"),
    "speechbrain": speechbrain.__version__,
    "gradio": gradio.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "soundfile": soundfile.__version__,
    "git_branch": GIT_BRANCH,
    "git_commit": GIT_COMMIT,
    "git_worktree_dirty": bool(GIT_STATUS),
}
(RUN_ROOT / "environment.json").write_text(
    json.dumps(environment, indent=2, ensure_ascii=False), encoding="utf-8"
)
display(pd.DataFrame(environment.items(), columns=["Componente", "Valor"]))


,Componente,Valor
0,run_id,20261104_082658
1,timestamp_utc,2026-11-04T08:27:02.031192+00:00
2,python,3.11.15
3,operating_system,Linux-7.3.1-1-cachyos-x86_64-with-glibc2.44
4,gpu,NVIDIA GeForce RTX 5060 Laptop GPU
5,vram_total_gib,7.616
6,torch,2.8.0+cu128
7,torch_cuda_available,True
8,cuda,12.8
9,transformers,4.57.3


## 3. Configuración experimental

Se fija la semilla. El modelo base es el valor por defecto del pipeline.
`TRANSLATION_MODEL = None` significa usar `configs/translation.yaml` sin
checkpoint controlado.


In [3]:
SEED = 42
OFFICIAL_SPLIT = "test"
N_SAMPLES = 20
TRANSLATION_MODEL = None
RUN_GRADIO_SMOKE = True
MAX_LENGTH = 128
NUM_BEAMS = 4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if CUDA_AVAILABLE:
    torch.cuda.manual_seed_all(SEED)

CONFIG = {
    "seed": SEED,
    "official_split": OFFICIAL_SPLIT,
    "n_samples": N_SAMPLES,
    "translation_model": TRANSLATION_MODEL or "config default (base)",
    "run_gradio_smoke": RUN_GRADIO_SMOKE,
    "max_length": MAX_LENGTH,
    "num_beams": NUM_BEAMS,
}
(RUN_ROOT / "config.json").write_text(
    json.dumps(CONFIG, indent=2, ensure_ascii=False), encoding="utf-8"
)
display(pd.DataFrame(CONFIG.items(), columns=["Parámetro", "Valor"]))


,Parámetro,Valor
0,seed,42
1,official_split,test
2,n_samples,20
3,translation_model,config default (base)
4,run_gradio_smoke,True
5,max_length,128
6,num_beams,4


## 4. Dataset B — reserva FLEURS oficial

La reserva vigente usa el split oficial `test`. Si no existe, se construye
filtrando audios vacíos, silenciosos o demasiado cortos. La reserva histórica
`v1` usaba `train` y no se usa aquí.


In [4]:
from src.evaluation.fleurs_benchmark import (
    build_official_reservation,
    load_reservation,
)

RESERVATION = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "manifests"
    / "fleurs_evaluation_v2_official_test.csv"
)
if not RESERVATION.exists():
    build_official_reservation(RESERVATION, n_samples=N_SAMPLES, split=OFFICIAL_SPLIT)

reservation = load_reservation(RESERVATION)
assert len(reservation) == N_SAMPLES
assert set(reservation["source_split"]) == {OFFICIAL_SPLIT}
display(
    reservation[
        ["sample_id", "sentence_id", "duration_sec", "sample_rate", "peak", "rms"]
    ].head(20)
)
print("Reserva:", RESERVATION.relative_to(PROJECT_ROOT))
print("Muestras:", len(reservation))
print("Duración total (s):", round(float(reservation["duration_sec"].sum()), 2))


,sample_id,sentence_id,duration_sec,sample_rate,peak,rms
0,fleurs_es_419_test_0000,2001,12.84,16000,0.712420,0.091004
1,fleurs_es_419_test_0001,1852,21.36,16000,0.528162,0.061353
2,fleurs_es_419_test_0002,1764,9.72,16000,0.667461,0.081045
3,fleurs_es_419_test_0003,1770,8.58,16000,0.292232,0.033261
4,fleurs_es_419_test_0004,1799,10.62,16000,0.358686,0.039022
5,fleurs_es_419_test_0005,1820,9.78,16000,0.449836,0.043072
6,fleurs_es_419_test_0006,1747,5.64,16000,0.376935,0.054540
7,fleurs_es_419_test_0007,1816,10.26,16000,0.298522,0.035162
8,fleurs_es_419_test_0008,1783,10.68,16000,0.505395,0.069874
9,fleurs_es_419_test_0009,1794,12.12,16000,0.404589,0.036373


Reserva: data/processed/manifests/fleurs_evaluation_v2_official_test.csv
Muestras: 20
Duración total (s): 228.12


## 5. Pipeline completo y métricas

Se ejecuta `process_audio` real para cada muestra: ASR, NMT base, XTTS y ECAPA.
La NMT aislada se calcula traduciendo la referencia española, sin errores de ASR.


In [5]:
from src.evaluation.fleurs_benchmark import run_benchmark, write_report

DATASET_B_DIR = RUN_ROOT / "datasetB"
aggregate = run_benchmark(
    RESERVATION,
    DATASET_B_DIR,
    translation_model_name_or_path=TRANSLATION_MODEL,
)
report_path = write_report(DATASET_B_DIR)
display(pd.DataFrame([aggregate]).T)
print("Reporte:", report_path.relative_to(PROJECT_ROOT))


Cargando modelo ASR (openai/whisper-small) en cuda...


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


fleurs_es_419_test_0000: WER=0.0345 ECAPA=0.5542 RTF=1.9804


fleurs_es_419_test_0001: WER=0.1569 ECAPA=0.4651 RTF=0.1891


fleurs_es_419_test_0002: WER=0.0909 ECAPA=0.4210 RTF=0.2083


... (15 outputs left out)

fleurs_es_419_test_0018: WER=0.0286 ECAPA=0.4439 RTF=0.1597


fleurs_es_419_test_0019: WER=0.1667 ECAPA=0.4863 RTF=0.1348


,0
dataset,google/fleurs
config,es_419
official_split,test
reservation,data/processed/manifests/fleurs_evaluation_v2_...
num_reserved,20
num_evaluated,20
num_excluded,0
wer_raw_mean,0.117339
wer_normalized_mean,0.061313
cer_mean,0.025203


Reporte: outputs/final_eval/master_20261104_082658/datasetB/REPORTE_DATASET_B.md


## 6. Resultados por muestra y propagación de errores

Se conservan las muestras excluidas (si las hay) y se identifican casos donde
Whisper difiere de la referencia española.


In [6]:
from src.evaluation.fleurs_benchmark import normalize_orthographic

results = pd.read_csv(DATASET_B_DIR / "results_per_sample.csv")
exclusions = pd.read_csv(DATASET_B_DIR / "exclusions.csv")
display(
    results[
        [
            "sample_id",
            "run_regime",
            "wer_raw",
            "wer_normalized",
            "cer",
            "speaker_similarity",
            "total_sec",
            "rtf",
        ]
    ].round(4)
)
display(exclusions)

error_rows = []
for row in results.itertuples(index=False):
    if normalize_orthographic(row.reference_es) != normalize_orthographic(
        row.transcription_es
    ):
        error_rows.append(
            {
                "sample_id": row.sample_id,
                "reference_es": row.reference_es,
                "whisper_es": row.transcription_es,
                "reference_en": row.reference_en,
                "pipeline_en": row.translation_en,
                "observacion": "La variación ASR se propaga a MarianMT y XTTS.",
            }
        )
error_frame = pd.DataFrame(error_rows)
error_frame.to_csv(
    DATASET_B_DIR / "error_propagation_examples.csv", index=False, encoding="utf-8"
)
display(error_frame.head(10))


,sample_id,run_regime,wer_raw,wer_normalized,cer,speaker_similarity,total_sec,rtf
0,fleurs_es_419_test_0000,cold,0.0345,0.0345,0.0058,0.5542,25.4282,1.9804
1,fleurs_es_419_test_0001,hot,0.1569,0.0980,0.0742,0.4651,4.0401,0.1891
2,fleurs_es_419_test_0002,hot,0.0909,0.0000,0.0174,0.4210,2.0244,0.2083
3,fleurs_es_419_test_0003,hot,0.2000,0.0000,0.0357,0.4028,1.0762,0.1254
4,fleurs_es_419_test_0004,hot,0.2143,0.2143,0.0194,0.3629,1.9302,0.1818
5,fleurs_es_419_test_0005,hot,0.3478,0.1304,0.0685,0.3505,1.7545,0.1794
6,fleurs_es_419_test_0006,hot,0.0000,0.0000,0.0000,0.3906,0.8946,0.1586
7,fleurs_es_419_test_0007,hot,0.0417,0.0417,0.0078,0.4920,1.7664,0.1722
8,fleurs_es_419_test_0008,hot,0.0000,0.0000,0.0000,0.4315,1.6257,0.1522
9,fleurs_es_419_test_0009,hot,0.1333,0.1000,0.0349,0.4466,2.1034,0.1736


,sample_id,sentence_id,reason


,sample_id,reference_es,whisper_es,reference_en,pipeline_en,observacion
0,fleurs_es_419_test_0000,Se recomienda enfáticamente a los viajeros que...,Se recomienda enfáticamente a los viajeros que...,Travellers are strongly advised to be aware of...,Travelers are strongly advised to report any e...,La variación ASR se propaga a MarianMT y XTTS.
1,fleurs_es_419_test_0001,El uso adecuado de los blogs «puede empoderar ...,El uso adecuado de los blogs pueden poderar a ...,"Appropriate use of blogs ""can empower students...",The proper use of blogs can empower students t...,La variación ASR se propaga a MarianMT y XTTS.
2,fleurs_es_419_test_0004,Carpanedo participó en dos carreras individual...,Carpaneo participó en dos carreras individuale...,"Beyond Wednesday's event, Carpanedo competed i...",Carpaneo participated in two individual champi...,La variación ASR se propaga a MarianMT y XTTS.
3,fleurs_es_419_test_0005,"Hoy en día, las personas escriben mensajes en ...",Hoy en día las personas escriben mensajes en l...,"People now write messages on computer screens,...",Nowadays people write messages on their comput...,La variación ASR se propaga a MarianMT y XTTS.
4,fleurs_es_419_test_0007,"Duvall, que está casado y tiene dos hijos adul...","Duval, que está casado y tiene dos hijos adult...","Duvall, who is married with two adult children...","Duval, who is married and has two adult childr...",La variación ASR se propaga a MarianMT y XTTS.
5,fleurs_es_419_test_0009,Se puede definir a una civilización como una c...,Se puede definir a una civilización como una c...,A civilization is a singular culture shared by...,A civilization can be defined as a specific cu...,La variación ASR se propaga a MarianMT y XTTS.
6,fleurs_es_419_test_0010,Las playas son casi todas de arena con lugares...,Las playas son casi todas de arena con lugares...,They are almost all sandy beaches with safe sw...,The beaches are almost all sandy with safe pla...,La variación ASR se propaga a MarianMT y XTTS.
7,fleurs_es_419_test_0011,No se ha emitido ni se emitirá ninguna alerta ...,No se ha emitido ni se emitirá ninguna alerta ...,"No tsunami warning has been issued, and accord...",No tsunami alert has been issued or will be is...,La variación ASR se propaga a MarianMT y XTTS.
8,fleurs_es_419_test_0012,No ha podido tomarse medicamentos para aliviar...,No ha podido tomarse medicamentos para liviar ...,He has been unable to take the drugs needed to...,You have not been able to take medicines to li...,La variación ASR se propaga a MarianMT y XTTS.
9,fleurs_es_419_test_0013,"Agregó: «Aun así, no es correcto exigirles que...","Agregó, aún así no es correcto exigirles que a...","He added that ""they should not, however, be as...","He added, it is still not correct to require t...",La variación ASR se propaga a MarianMT y XTTS.


## 7. Smoke test de Gradio

Se construye la interfaz, se lanza en un puerto local, se procesa una muestra
real con `_run_demo` y se cierra el servidor. No se deja abierto.


In [7]:
gradio_evidence = {"executed": False}
if RUN_GRADIO_SMOKE:
    from src.pipeline.demo import _run_demo, create_demo

    sample_audio = sorted((DATASET_B_DIR / "audio_original").glob("*.wav"))[0]
    gradio_result = _run_demo(str(sample_audio))
    demo = create_demo()
    demo.launch(
        server_name="127.0.0.1",
        server_port=None,
        prevent_thread_lock=True,
        quiet=True,
    )
    launched = demo.server is not None
    demo.close()
    gradio_evidence = {
        "executed": True,
        "sample": sample_audio.name,
        "transcription_es": gradio_result[0],
        "translation_en": gradio_result[1],
        "output_exists": Path(gradio_result[2]).exists(),
        "total_time_label": gradio_result[3],
        "speaker_similarity": gradio_result[4],
        "server_launched": launched,
    }
    (RUN_ROOT / "gradio_smoke.json").write_text(
        json.dumps(gradio_evidence, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    assert launched and gradio_evidence["output_exists"]
display(pd.DataFrame([gradio_evidence]).T)


Closing server running on port: 7860


,0
executed,True
sample,fleurs_es_419_test_0000.wav
transcription_es,Se recomienda enfáticamente a los viajeros que...
translation_en,Travelers are strongly advised to report any e...
output_exists,True
total_time_label,1.90 s
speaker_similarity,0.5518
server_launched,True


## 8. Pruebas de entradas inválidas

Se verifica que el pipeline rechaza audio silencioso, audio demasiado corto y
archivo inexistente antes de cargar modelos. También se comprueba que XTTS
rechaza texto vacío.


In [8]:
import tempfile

from src.pipeline.runner import process_audio
from src.voice_cloning.synthesize import synthesize

invalid_checks = {}
with tempfile.TemporaryDirectory(prefix="invalid_s2s_") as tmp:
    tmp = Path(tmp)
    silent = tmp / "silent.wav"
    short = tmp / "short.wav"
    soundfile.write(silent, np.zeros(16000, dtype=np.float32), 16000, subtype="FLOAT")
    soundfile.write(short, np.full(800, 0.1, dtype=np.float32), 16000, subtype="FLOAT")

    for name, audio_path in (
        ("silent", silent),
        ("short", short),
        ("missing", tmp / "missing.wav"),
    ):
        try:
            process_audio(str(audio_path), str(tmp / f"out_{name}"))
        except Exception as error:
            invalid_checks[name] = {
                "raised": type(error).__name__,
                "message": str(error),
            }
        else:
            invalid_checks[name] = {"raised": None, "message": "NO SE LANZÓ ERROR"}

    try:
        synthesize("", str(silent), str(tmp / "empty.wav"), language="en")
    except Exception as error:
        invalid_checks["empty_text"] = {
            "raised": type(error).__name__,
            "message": str(error),
        }
    else:
        invalid_checks["empty_text"] = {
            "raised": None,
            "message": "NO SE LANZÓ ERROR",
        }

assert all(item["raised"] for item in invalid_checks.values())
(RUN_ROOT / "invalid_input_checks.json").write_text(
    json.dumps(invalid_checks, indent=2, ensure_ascii=False), encoding="utf-8"
)
display(pd.DataFrame(invalid_checks).T)


,raised,message
silent,ValueError,El audio parece silencioso (RMS=0.00000000 < 0...
short,ValueError,El audio es demasiado corto (0.050 s < 0.500 s...
missing,FileNotFoundError,Archivo de entrada no encontrado: missing.wav
empty_text,ValueError,El texto a sintetizar no puede estar vacío.


## 9. Experimento NMT de 10 000 pares

No se reentrena. Se carga el resumen sellado del experimento controlado y se
documenta la decisión de usar MarianMT base en el pipeline.


In [9]:
nmt_summary_path = PROJECT_ROOT / "Docs" / "NMT_CONTROLLED_EXPERIMENT_10K.json"
nmt_summary = json.loads(nmt_summary_path.read_text(encoding="utf-8"))
nmt_table = pd.DataFrame(
    [
        {
            "Métrica": "SacreBLEU",
            "Base": nmt_summary["base"]["sacrebleu"],
            "Controlled 10k": nmt_summary["controlled"]["sacrebleu"],
            "Delta": nmt_summary["delta"]["sacrebleu"],
        },
        {
            "Métrica": "chrF",
            "Base": nmt_summary["base"]["chrf"],
            "Controlled 10k": nmt_summary["controlled"]["chrf"],
            "Delta": nmt_summary["delta"]["chrf"],
        },
        {
            "Métrica": "Eval loss",
            "Base": nmt_summary["base"]["eval_loss"],
            "Controlled 10k": nmt_summary["controlled"]["eval_loss"],
            "Delta": nmt_summary["delta"]["eval_loss"],
        },
    ]
)
display(nmt_table.round(4))
print("Decisión:", nmt_summary["decision"])
print("Pares usados:", nmt_summary["training"]["train_used"])
print("Épocas:", nmt_summary["training"]["epochs"])


,Métrica,Base,Controlled 10k,Delta
0,SacreBLEU,43.4140,42.6499,-0.7641
1,chrF,64.4179,63.6674,-0.7506
2,Eval loss,1.3532,1.4047,0.0514


Decisión: Use MarianMT base in the pipeline; do not present the controlled checkpoint as an improvement.
Pares usados: 10000
Épocas: 1


## 10. Exportación de evidencia

Se genera un resumen y un ZIP con JSON, CSV, Markdown y figuras ligeras. No se
incluyen WAV ni checkpoints para evitar un paquete innecesariamente pesado.


In [10]:
import zipfile

runtime_total_seconds = time.perf_counter() - NOTEBOOK_STARTED if "NOTEBOOK_STARTED" in globals() else None
execution_summary = {
    "run_id": RUN_ID,
    "git_branch": GIT_BRANCH,
    "git_commit": GIT_COMMIT,
    "git_worktree_dirty": bool(GIT_STATUS),
    "runtime_total_seconds": runtime_total_seconds,
    "dataset_b": aggregate,
    "gradio": gradio_evidence,
    "invalid_inputs": invalid_checks,
    "nmt_controlled_10k": nmt_summary,
    "limitations": [
        "MOS no realizado.",
        "Clonación aproximada; no garantiza emociones ni prosodia.",
        "No es transmisión en tiempo real.",
        "FLEURS no expone speaker_id; no se mide leakage de hablante.",
        "XTTS-v2/Coqui usa CPML; revisar licencia antes de distribuir pesos.",
    ],
}
(RUN_ROOT / "execution_summary.json").write_text(
    json.dumps(execution_summary, indent=2, ensure_ascii=False), encoding="utf-8"
)

report = f'''# MASTER COLAB EXECUTION REPORT

- RUN_ID: `{RUN_ID}`
- Commit: `{GIT_COMMIT}`
- GPU: `{environment['gpu']}`
- Python: `{environment['python']}`
- Split FLEURS: `{aggregate['official_split']}`
- Muestras evaluadas: `{aggregate['num_evaluated']}`
- Muestras excluidas: `{aggregate['num_excluded']}`

## Dataset B

| Métrica | Valor |
|---|---:|
| WER raw | {aggregate['wer_raw_mean']:.6f} |
| WER normalizado | {aggregate['wer_normalized_mean']:.6f} |
| CER | {aggregate['cer_mean']:.6f} |
| BLEU aislado | {aggregate['sacrebleu_isolated']:.4f} |
| chrF aislado | {aggregate['chrf_isolated']:.4f} |
| BLEU pipeline | {aggregate['sacrebleu_pipeline']:.4f} |
| chrF pipeline | {aggregate['chrf_pipeline']:.4f} |
| ECAPA media | {aggregate['speaker_similarity_mean']:.4f} |
| RTF hot medio | {aggregate['rtf_hot_mean']} |

## NMT 10k

El checkpoint controlado obtuvo BLEU {nmt_summary['controlled']['sacrebleu']:.4f} frente a {nmt_summary['base']['sacrebleu']:.4f} del base. No se presenta como mejora; el pipeline usa base.

## Gradio

- Ejecutado: `{gradio_evidence.get('executed')}`
- Servidor lanzado: `{gradio_evidence.get('server_launched')}`
- Salida generada: `{gradio_evidence.get('output_exists')}`

## Limitaciones

- MOS no realizado.
- Clonación aproximada.
- Sin transmisión en tiempo real.
- Licencia CPML de XTTS-v2.
'''
(RUN_ROOT / "MASTER_REPORT.md").write_text(report, encoding="utf-8")

zip_path = RUN_ROOT.parent / f"Proyecto_IA_MASTER_EVIDENCIAS_{RUN_ID}.zip"
allowed_suffixes = {".json", ".csv", ".md", ".png", ".txt", ".log"}
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in RUN_ROOT.rglob("*"):
        if path.is_file() and path.suffix.lower() in allowed_suffixes:
            archive.write(path, path.relative_to(RUN_ROOT.parent).as_posix())

print("Resumen:", (RUN_ROOT / "execution_summary.json").relative_to(PROJECT_ROOT))
print("Reporte:", (RUN_ROOT / "MASTER_REPORT.md").relative_to(PROJECT_ROOT))
print(
    "ZIP:",
    zip_path.relative_to(PROJECT_ROOT),
    f"({zip_path.stat().st_size / 2**20:.3f} MiB)",
)
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))


Resumen: outputs/final_eval/master_20261104_082658/execution_summary.json
Reporte: outputs/final_eval/master_20261104_082658/MASTER_REPORT.md
ZIP: outputs/final_eval/Proyecto_IA_MASTER_EVIDENCIAS_20261104_082658.zip (0.045 MiB)


## 11. Conclusiones y límites

- El pipeline completo se ejecutó con modelos reales sobre el split oficial `test` de FLEURS `es_419`.
- La reserva oficial y el filtro de audio corrigen el error de la reserva histórica basada en `train`.
- MarianMT base es la opción del pipeline. El ajuste de 10 000 pares no mejoró BLEU/chrF y se documenta como experimento, no como mejora.
- Gradio se lanzó y procesó una muestra; las entradas inválidas se rechazan antes de cargar modelos.
- No se realizó MOS ni entrenamiento de 100 000 pares.
- La clonación es aproximada y no garantiza emociones, prosodia ni identidad perfecta.
- XTTS-v2/Coqui usa CPML; la licencia MIT del código no cubre los pesos.
